**AI-Based Course Information Retrieval from Vignan University Curriculum**

In [1]:
!pip install -q pypdf sentence-transformers faiss-cpu transformers accelerate

In [2]:
from google.colab import files

uploaded = files.upload()

pdf_file = list(uploaded.keys())[0]

print("PDF uploaded successfully!")
print("File name:", pdf_file)

Saving R22 B.Tech (CSE-AI&ML) Course Structure and Contents.pdf to R22 B.Tech (CSE-AI&ML) Course Structure and Contents (1).pdf
PDF uploaded successfully!
File name: R22 B.Tech (CSE-AI&ML) Course Structure and Contents (1).pdf


In [3]:
from pypdf import PdfReader

reader = PdfReader(pdf_file)

print("Total pages:", len(reader.pages))

Total pages: 188


In [4]:
pages = []

for page_number, page in enumerate(reader.pages, start=1):

    text = page.extract_text()

    if text:
        pages.append({
            "page": page_number,
            "text": text
        })

print("Pages with extracted text:", len(pages))

Pages with extracted text: 179


In [5]:
for page in pages[:3]:

    print("=" * 80)
    print("PAGE:", page["page"])
    print(page["text"][:1500])

PAGE: 1
B.Tech. ELECTRICAL AND ELECTRONICS ENGINEERING
TABLE OF CONTENTS
    Page Number
Foreword    xxix
VFSTR - Vision & Mission   xxxi
EEE - Vision & Mission   xxxi
Programme - Educational Objectives, Outcomes, Specific Outcomes  xxxi
Curriculum Structure   xxxii
Course Contents
I YEAR - I SEMESTER
 21HS105 Engineering Mathematics - I (E) 3
 21HS113 Engineering Physics (A)  5
 21EE101 Basic Electrical and Electronics Engineering 8
 21ME104 Engineering Graphics Laboratory 10
 21CS151 Introduction to C Programming 14
I YEAR - II SEMESTER
 21HS111 Engineering Mathematics - II (E) 18
 21HS118 Engineering Chemistry (C) 20
 21CS152 Programming for Problem Solving 22
 21HS122 English Proficiency and Communication Skills  27
 21HS123 Technical English Communication  29
 21HS124 Constitution of India  32
 21EE102 Basic Engineering Products  34
 21ME103 Workshop  37
II YEAR - I SEMESTER
 21CS251 Data Structures 41
 21EE201 Electrical Circuit Analysis  45
 21EE202 Digital Electronic Circuits  

In [6]:
import re

def clean_text(text):

    text = text.replace("\n", " ")

    text = re.sub(r'\s+', ' ', text)

    return text.strip()


for page in pages:
    page["text"] = clean_text(page["text"])

print(pages[0]["text"][:1000])

B.Tech. ELECTRICAL AND ELECTRONICS ENGINEERING TABLE OF CONTENTS Page Number Foreword xxix VFSTR - Vision & Mission xxxi EEE - Vision & Mission xxxi Programme - Educational Objectives, Outcomes, Specific Outcomes xxxi Curriculum Structure xxxii Course Contents I YEAR - I SEMESTER 21HS105 Engineering Mathematics - I (E) 3 21HS113 Engineering Physics (A) 5 21EE101 Basic Electrical and Electronics Engineering 8 21ME104 Engineering Graphics Laboratory 10 21CS151 Introduction to C Programming 14 I YEAR - II SEMESTER 21HS111 Engineering Mathematics - II (E) 18 21HS118 Engineering Chemistry (C) 20 21CS152 Programming for Problem Solving 22 21HS122 English Proficiency and Communication Skills 27 21HS123 Technical English Communication 29 21HS124 Constitution of India 32 21EE102 Basic Engineering Products 34 21ME103 Workshop 37 II YEAR - I SEMESTER 21CS251 Data Structures 41 21EE201 Electrical Circuit Analysis 45 21EE202 Digital Electronic Circuits 47 21EE203 Analog Electronics 49 21EE204 Elect

In [7]:
def create_chunks(text, chunk_size=800, overlap=100):

    chunks = []

    start = 0

    while start < len(text):

        end = start + chunk_size

        chunk = text[start:end]

        if chunk.strip():
            chunks.append(chunk)

        start += chunk_size - overlap

    return chunks


documents = []

for page in pages:

    chunks = create_chunks(page["text"])

    for chunk_id, chunk in enumerate(chunks):

        documents.append({
            "text": chunk,
            "page": page["page"],
            "chunk_id": chunk_id,
            "source": pdf_file
        })


print("Total chunks created:", len(documents))

Total chunks created: 669


In [8]:
for doc in documents[:3]:

    print("=" * 80)

    print("Page:", doc["page"])
    print("Chunk:", doc["chunk_id"])

    print(doc["text"][:700])

Page: 1
Chunk: 0
B.Tech. ELECTRICAL AND ELECTRONICS ENGINEERING TABLE OF CONTENTS Page Number Foreword xxix VFSTR - Vision & Mission xxxi EEE - Vision & Mission xxxi Programme - Educational Objectives, Outcomes, Specific Outcomes xxxi Curriculum Structure xxxii Course Contents I YEAR - I SEMESTER 21HS105 Engineering Mathematics - I (E) 3 21HS113 Engineering Physics (A) 5 21EE101 Basic Electrical and Electronics Engineering 8 21ME104 Engineering Graphics Laboratory 10 21CS151 Introduction to C Programming 14 I YEAR - II SEMESTER 21HS111 Engineering Mathematics - II (E) 18 21HS118 Engineering Chemistry (C) 20 21CS152 Programming for Problem Solving 22 21HS122 English Proficiency and Communication Skills 27 21H
Page: 1
Chunk: 1
S123 Technical English Communication 29 21HS124 Constitution of India 32 21EE102 Basic Engineering Products 34 21ME103 Workshop 37 II YEAR - I SEMESTER 21CS251 Data Structures 41 21EE201 Electrical Circuit Analysis 45 21EE202 Digital Electronic Circuits 47 21EE203 

In [9]:
def detect_semester(text):

    patterns = [
        r'(\d+)(?:st|nd|rd|th)\s+semester',
        r'semester\s*[-:]?\s*(\d+)'
    ]

    for pattern in patterns:

        match = re.search(
            pattern,
            text,
            re.IGNORECASE
        )

        if match:
            return int(match.group(1))

    return None


for doc in documents:

    doc["semester"] = detect_semester(doc["text"])


print("Metadata extraction completed!")

Metadata extraction completed!


In [10]:
for doc in documents[:10]:

    print(
        "Page:",
        doc["page"],
        "| Semester:",
        doc["semester"]
    )

Page: 1 | Semester: 21
Page: 1 | Semester: 21
Page: 1 | Semester: 22
Page: 1 | Semester: 22
Page: 1 | Semester: None
Page: 2 | Semester: 22
Page: 2 | Semester: 22
Page: 2 | Semester: None
Page: 3 | Semester: None
Page: 3 | Semester: None


In [11]:
current_semester = None

for doc in documents:

    detected = detect_semester(doc["text"])

    if detected is not None:
        current_semester = detected

    doc["semester"] = current_semester


for doc in documents[:15]:

    print(
        "Page:", doc["page"],
        "| Semester:", doc["semester"]
    )

Page: 1 | Semester: 21
Page: 1 | Semester: 21
Page: 1 | Semester: 22
Page: 1 | Semester: 22
Page: 1 | Semester: 22
Page: 2 | Semester: 22
Page: 2 | Semester: 22
Page: 2 | Semester: 22
Page: 3 | Semester: 22
Page: 3 | Semester: 22
Page: 3 | Semester: 22
Page: 3 | Semester: 22
Page: 3 | Semester: 22
Page: 4 | Semester: 22
Page: 4 | Semester: 22


In [12]:
def detect_course_code(text):

    patterns = [
        r'\b\d{2}[A-Z]{2,5}\d{3}\b',
        r'\b[A-Z]{2,5}\d{3,4}\b'
    ]

    for pattern in patterns:

        match = re.search(pattern, text)

        if match:
            return match.group(0)

    return None


for doc in documents:

    doc["course_code"] = detect_course_code(
        doc["text"]
    )


print("Course code extraction completed!")

Course code extraction completed!


In [13]:
for doc in documents:

    if doc["course_code"]:

        print(
            "Course Code:",
            doc["course_code"],
            "| Page:",
            doc["page"]
        )

Course Code: 21HS105 | Page: 1
Course Code: 21HS124 | Page: 1
Course Code: 22MT103 | Page: 1
Course Code: 22ME101 | Page: 1
Course Code: 22AM203 | Page: 1
Course Code: 22TP301 | Page: 2
Course Code: 22AM403 | Page: 2
Course Code: 22AM952 | Page: 2
Course Code: 22MT103 | Page: 6
Course Code: 22MT105 | Page: 6
Course Code: 22ST203 | Page: 7
Course Code: 22TP203 | Page: 7
Course Code: 22TP301 | Page: 8
Course Code: 22TP302 | Page: 8
Course Code: 22AM401 | Page: 9
Course Code: 22AM801 | Page: 10
Course Code: 22AM951 | Page: 10
Course Code: 22MT103 | Page: 11
Course Code: 22CS104 | Page: 11
Course Code: 22MT103 | Page: 13
Course Code: 22PY105 | Page: 15
Course Code: 22EE101 | Page: 17
Course Code: 22CS103 | Page: 19
Course Code: 22TP103 | Page: 22
Course Code: 22EN102 | Page: 32
Course Code: 22TP101 | Page: 34
Course Code: 22MT105 | Page: 36
Course Code: 22MT107 | Page: 38
Course Code: 22ME101 | Page: 40
Course Code: 22TP104 | Page: 42
Course Code: 22EN104 | Page: 51
Course Code: 22CS104 | 

In [14]:
def detect_credits(text):

    patterns = [
        r'credits?\s*[:\-]?\s*(\d+(?:\.\d+)?)',
        r'\bC\s*[:\-]?\s*(\d+(?:\.\d+)?)\b'
    ]

    for pattern in patterns:

        match = re.search(
            pattern,
            text,
            re.IGNORECASE
        )

        if match:
            return float(match.group(1))

    return None


for doc in documents:

    doc["credits"] = detect_credits(
        doc["text"]
    )


print("Credits extraction completed!")

Credits extraction completed!


In [15]:
for doc in documents[:15]:

    print("-" * 70)

    print("Course Code :", doc["course_code"])
    print("Semester    :", doc["semester"])
    print("Credits     :", doc["credits"])
    print("Page        :", doc["page"])

----------------------------------------------------------------------
Course Code : 21HS105
Semester    : 21
Credits     : None
Page        : 1
----------------------------------------------------------------------
Course Code : 21HS124
Semester    : 21
Credits     : None
Page        : 1
----------------------------------------------------------------------
Course Code : 22MT103
Semester    : 22
Credits     : 22.0
Page        : 1
----------------------------------------------------------------------
Course Code : 22ME101
Semester    : 22
Credits     : None
Page        : 1
----------------------------------------------------------------------
Course Code : 22AM203
Semester    : 22
Credits     : None
Page        : 1
----------------------------------------------------------------------
Course Code : 22TP301
Semester    : 22
Credits     : None
Page        : 2
----------------------------------------------------------------------
Course Code : 22AM403
Semester    : 22
Credits     : None
P

In [16]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

texts = [
    doc["text"]
    for doc in documents
]

embeddings = embedding_model.encode(
    texts,
    show_progress_bar=True
)

print("Embedding creation completed!")
print("Number of embeddings:", len(embeddings))
print("Embedding size:", embeddings.shape[1])

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/21 [00:00<?, ?it/s]

Embedding creation completed!
Number of embeddings: 669
Embedding size: 384


In [17]:
import faiss
import numpy as np

embedding_array = np.array(
    embeddings
).astype("float32")

dimension = embedding_array.shape[1]

index = faiss.IndexFlatL2(
    dimension
)

index.add(embedding_array)

print("FAISS vector database created!")
print("Total vectors:", index.ntotal)

FAISS vector database created!
Total vectors: 669


In [18]:
def semantic_search(query, k=5):

    query_embedding = embedding_model.encode(
        [query]
    )

    query_embedding = np.array(
        query_embedding
    ).astype("float32")

    distances, indices = index.search(
        query_embedding,
        k
    )

    results = []

    for distance, idx in zip(
        distances[0],
        indices[0]
    ):

        result = documents[idx].copy()

        result["distance"] = float(distance)

        results.append(result)

    return results

In [19]:
results = semantic_search(
    "What subjects are taught in the curriculum?"
)

for result in results:

    print("=" * 70)
    print("Page:", result["page"])
    print(result["text"][:500])

Page: 111
 the class. Students would have to solve 10 additional problems as homework assignments in each concept. Source: https:// images.app.goo.gl/ kvtVgA8TkvDCqLhj7
Page: 8
/ Honours - 2 3 0 2 4 Total 18 2 12 25 32 Hrs 25 III Year II Semester Course Code Course Title L T P C Course category 22TP302 Quantitative Aptitude and Logical Reasoning 1 2 0 2 Humanities 22AM304 Fundamentals of Image Processing 2 0 2 3 Professional Core 22AM305 Reinforcement Learning 2 2 0 3 Professional Core 22CS303 Web Technologies 2 0 4 4 Professional Core 22AM306 Inter-Disciplinary Project – Phase II 0 0 2 2 Project Department Elective – 2 3 0 2 4 Department Elective Open Elective – 3 3 0
Page: 8
4 R22 B.Tech. YEAR DEGREE PROGRAMME VFSTR 8 Course Structure COURSE STRUCTURE - R22 III Year I Semester Course Code Course Title L T P C Course category 22TP301 Soft Skills Laboratory 0 0 2 1 Humanities 22AM301 Deep Learning 3 0 2 4 Professional Core 22CS204 Computer Networks 3 0 2 4 Professional Core 22DS203 For

In [20]:
def filter_by_metadata(
    semester=None,
    course_code=None,
    credits=None
):

    filtered = []

    for doc in documents:

        if semester is not None:
            if doc["semester"] != semester:
                continue

        if course_code is not None:
            if doc["course_code"] != course_code:
                continue

        if credits is not None:
            if doc["credits"] != credits:
                continue

        filtered.append(doc)

    return filtered

In [21]:
semester_3_docs = filter_by_metadata(
    semester=3
)

print(
    "Documents belonging to Semester 3:",
    len(semester_3_docs)
)

Documents belonging to Semester 3: 15


In [22]:
for doc in semester_3_docs:

    print("=" * 70)

    print("Page:", doc["page"])
    print("Course Code:", doc["course_code"])
    print("Credits:", doc["credits"])

    print(doc["text"][:500])

Page: 24
Course Code: None
Credits: None
VFSTR 24 CSE - AI & ML I Year I Semester 3. Write a program to accept a number N as input from the user and print the following pattern. Sample N = 5. * ** *** **** ***** 4. Write a program to accept a number N as input from the user and print the following pattern. Sample N = 5. * ** *** **** ***** 5. Write a program to accept a number N as input from the user and print the following pattern. Sample N = 5. 1 12 123 1234 12345 6. Write a program to accept a number N as input from the user and pr
Page: 24
Course Code: None
Credits: None
a program to accept a number N as input from the user and print the following pattern. Sample N = 5. 12345 2345 345 45 5 9. Write a program to accept a number N as input from the user and print the following pattern. Sample N = 5. A AB ABC ABCD ABCDE
Page: 46
Course Code: None
Credits: None
VFSTR 46 CSE - AI & ML I Year II Semester 3*3*3 4*4*4*4 4*4*4*4 3*3*3 2*2 1 ● Write a program to generate the following patte

In [23]:
def metadata_search(
    query,
    semester=None,
    course_code=None,
    credits=None,
    k=5
):

    filtered_docs = filter_by_metadata(
        semester=semester,
        course_code=course_code,
        credits=credits
    )

    if not filtered_docs:
        return []

    filtered_texts = [
        doc["text"]
        for doc in filtered_docs
    ]

    filtered_embeddings = embedding_model.encode(
        filtered_texts
    )

    filtered_embeddings = np.array(
        filtered_embeddings
    ).astype("float32")

    query_embedding = embedding_model.encode(
        [query]
    )

    query_embedding = np.array(
        query_embedding
    ).astype("float32")

    temp_index = faiss.IndexFlatL2(
        filtered_embeddings.shape[1]
    )

    temp_index.add(filtered_embeddings)

    k = min(k, len(filtered_docs))

    distances, indices = temp_index.search(
        query_embedding,
        k
    )

    results = []

    for distance, idx in zip(
        distances[0],
        indices[0]
    ):

        result = filtered_docs[idx].copy()

        result["distance"] = float(distance)

        results.append(result)

    return results

In [24]:
results = metadata_search(
    "What subjects are available?",
    semester=3,
    k=10
)

print("Retrieved documents:", len(results))

for result in results:

    print("\n" + "=" * 70)

    print("Course Code:", result["course_code"])
    print("Semester:", result["semester"])
    print("Credits:", result["credits"])
    print("Page:", result["page"])

Retrieved documents: 10

Course Code: None
Semester: 3
Credits: None
Page: 64

Course Code: None
Semester: 3
Credits: None
Page: 63

Course Code: None
Semester: 3
Credits: None
Page: 64

Course Code: None
Semester: 3
Credits: None
Page: 48

Course Code: None
Semester: 3
Credits: None
Page: 47

Course Code: None
Semester: 3
Credits: None
Page: 48

Course Code: None
Semester: 3
Credits: None
Page: 48

Course Code: None
Semester: 3
Credits: None
Page: 48

Course Code: None
Semester: 3
Credits: None
Page: 46

Course Code: None
Semester: 3
Credits: None
Page: 46


In [25]:
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM
)

import torch

model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(
    model_name
)

model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name
)

device = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

model = model.to(device)

print("Model loaded successfully!")
print("Device:", device)

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Model loaded successfully!
Device: cpu


In [26]:
def generate_answer(question, context):

    prompt = f"""
You are a college course advisor.

Answer the question ONLY using the
provided curriculum information.

Do not invent information.

If the information is not available,
say:
"Information not found in the curriculum."

Question:
{question}

Curriculum:
{context}

Answer:
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=4096
    )

    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    output = model.generate(
        **inputs,
        max_new_tokens=300
    )

    answer = tokenizer.decode(
        output[0],
        skip_special_tokens=True
    )

    return answer

In [27]:
def add_sources(answer, results):

    source_text = "\n\nSources:\n"

    used = set()

    for result in results:

        page = result["page"]

        if page not in used:

            source_text += (
                f"- {result['source']} "
                f"(Page {page})\n"
            )

            used.add(page)

    return answer + source_text

In [30]:
def build_context(results):
    context = ""
    for result in results:
        context += result["text"] + "\n\n"
    return context.strip()

def course_advisor(
    question,
    semester=None,
    course_code=None,
    credits=None,
    k=5
):

    results = metadata_search(
        query=question,
        semester=semester,
        course_code=course_code,
        credits=credits,
        k=k
    )

    if not results:

        return (
            "Information not found in "
            "the curriculum."
        )

    context = build_context(results)

    answer = generate_answer(
        question,
        context
    )

    final_answer = add_sources(
        answer,
        results
    )

    return final_answer

In [31]:
question = "What subjects are available in the third semester?"

answer = course_advisor(
    question,
    semester=3,
    k=10
)

print(answer)

s = input() a = s.count('0') b = s.count('1') c = s.count('2') if a == 0: print(0) else: print('0')

Sources:
- R22 B.Tech (CSE-AI&ML) Course Structure and Contents (1).pdf (Page 64)
- R22 B.Tech (CSE-AI&ML) Course Structure and Contents (1).pdf (Page 63)
- R22 B.Tech (CSE-AI&ML) Course Structure and Contents (1).pdf (Page 48)
- R22 B.Tech (CSE-AI&ML) Course Structure and Contents (1).pdf (Page 47)
- R22 B.Tech (CSE-AI&ML) Course Structure and Contents (1).pdf (Page 46)



In [32]:
question = "What are the credits for Data Science subjects?"

answer = course_advisor(
    question,
    k=10
)

print(answer)

B.Tech. IV Y E A R I SEM & II SEM COURSE CONTENTS I SEMESTER  22AM401 - Knowledge Representation and Reasoning  22AM402 - Text Mining  22CS402 - Big Data Analytics  - Department Elective - 3  - Department Elective - 4 II SEMESTER  22AM403 - Internship / Project Work CSE - ARTIFICIAL INTELLIGENCE AND MACHINE LEARNING VFSTR 145 CSE - AI & ML - Department Electives

Sources:
- R22 B.Tech (CSE-AI&ML) Course Structure and Contents (1).pdf (Page 121)
- R22 B.Tech (CSE-AI&ML) Course Structure and Contents (1).pdf (Page 145)
- R22 B.Tech (CSE-AI&ML) Course Structure and Contents (1).pdf (Page 7)
- R22 B.Tech (CSE-AI&ML) Course Structure and Contents (1).pdf (Page 2)
- R22 B.Tech (CSE-AI&ML) Course Structure and Contents (1).pdf (Page 8)
- R22 B.Tech (CSE-AI&ML) Course Structure and Contents (1).pdf (Page 150)
- R22 B.Tech (CSE-AI&ML) Course Structure and Contents (1).pdf (Page 9)
- R22 B.Tech (CSE-AI&ML) Course Structure and Contents (1).pdf (Page 161)
- R22 B.Tech (CSE-AI&ML) Course Structure

In [33]:
question = "Which courses have programming prerequisites?"

answer = course_advisor(
    question,
    k=10
)

print(answer)

ay Mittal, “Programming in C - A Practical Approach”, 1st edition, Pearson Education, India, 2010.

Sources:
- R22 B.Tech (CSE-AI&ML) Course Structure and Contents (1).pdf (Page 50)
- R22 B.Tech (CSE-AI&ML) Course Structure and Contents (1).pdf (Page 7)
- R22 B.Tech (CSE-AI&ML) Course Structure and Contents (1).pdf (Page 58)
- R22 B.Tech (CSE-AI&ML) Course Structure and Contents (1).pdf (Page 6)
- R22 B.Tech (CSE-AI&ML) Course Structure and Contents (1).pdf (Page 31)
- R22 B.Tech (CSE-AI&ML) Course Structure and Contents (1).pdf (Page 92)
- R22 B.Tech (CSE-AI&ML) Course Structure and Contents (1).pdf (Page 137)
- R22 B.Tech (CSE-AI&ML) Course Structure and Contents (1).pdf (Page 8)
- R22 B.Tech (CSE-AI&ML) Course Structure and Contents (1).pdf (Page 2)



In [35]:
while True:

    question = input(
        "\nAsk your course question "
        "(type 'exit' to stop): "
    )

    if question.lower() == "exit":
        print("Course Advisor stopped.")
        break

    answer = course_advisor(
        question,
        k=5
    )

    print("\nAI COURSE ADVISOR:")
    print(answer)


Ask your course question (type 'exit' to stop): aiml

AI COURSE ADVISOR:
Information not found in the curriculum.

Sources:
- R22 B.Tech (CSE-AI&ML) Course Structure and Contents (1).pdf (Page 3)
- R22 B.Tech (CSE-AI&ML) Course Structure and Contents (1).pdf (Page 101)
- R22 B.Tech (CSE-AI&ML) Course Structure and Contents (1).pdf (Page 159)
- R22 B.Tech (CSE-AI&ML) Course Structure and Contents (1).pdf (Page 156)
- R22 B.Tech (CSE-AI&ML) Course Structure and Contents (1).pdf (Page 11)


Ask your course question (type 'exit' to stop): exit
Course Advisor stopped.
